# S&P 500 All Assets ETL Pipeline

## Purpose
This notebook implements an ETL pipeline that:
- Fetches daily S&P 500 stock data from Kaggle (OHLCV data for all assets)
- Transforms wide-format data to normalized long-format (ticker-level granularity)
- Loads data into Delta Lake for time-series analysis

## Architecture
- **Source**: Kaggle dataset (yash16jr/s-and-p500-daily-update-dataset)
- **Target**: Delta table `thekitchen.kg.sp500_aa_daily`
- **Pattern**: Full refresh (overwrite) with structured OHLCV data
- **Compute**: Databricks Serverless (Python + PySpark)

## Workflow
1. Install Kaggle SDK
2. Authenticate via Job Parameters (secure token management)
3. Download dataset from Kaggle
4. Transform: Wide → Long format (unpivot OHLCV metrics)
5. Load to Delta Lake

## Data Source
**Link**: https://www.kaggle.com/datasets/yash16jr/s-and-p500-daily-update-dataset

In [0]:
# ==============================================================================
# STEP 1: Install Kaggle SDK with pandas support
# ==============================================================================
# Package: kagglehub[pandas-datasets] - Official Kaggle Python SDK
# Purpose: Direct dataset access without manual downloads
# Note: Requires kernel restart after installation

%pip install kagglehub[pandas-datasets]


In [0]:
# ==============================================================================
# STEP 2: Authenticate with Kaggle API (Secure Token Management)
# ==============================================================================
# Pattern: Job Parameter Widget (avoids hardcoding credentials)
# Token: Generated from Kaggle Account Settings → Create New API Token
# Usage: Set widget value in Databricks Job configuration
# Alternative: Use dbutils.secrets.get() for production

import os

try:
    # Retrieve API token securely from Job Parameter Widget
    kaggle_token = dbutils.widgets.get("KAGGLE_API_TOKEN")
    
    # Set as environment variable for Kaggle library
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    
    print("✓ Kaggle API Token successfully loaded from Job Parameters")
except Exception as e:
    print("⚠ Error: Could not find 'KAGGLE_API_TOKEN' parameter. ")
    print("  Please add it in the Job Details or use dbutils.secrets.get() instead.")



In [0]:
# ==============================================================================
# STEP 3: Download S&P 500 dataset from Kaggle
# ==============================================================================
# Dataset: Daily OHLCV data for all S&P 500 constituents
# Format: Wide format (1 row per date, multiple ticker columns)
# Adapter: PANDAS (direct pandas DataFrame)
# Note: First run downloads data, subsequent runs use cache

import kagglehub
from kagglehub import KaggleDatasetAdapter

df_pd = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yash16jr/s-and-p500-daily-update-dataset",
    "SnP_daily_update.csv",
)

print(f"✓ Pandas DF Shape: {df_pd.shape[0]:,} rows × {df_pd.shape[1]:,} columns")
df_pd.head()


In [0]:
# ==============================================================================
# STEP 4: Transform from Wide to Long Format (Unpivot OHLCV)
# ==============================================================================
# Challenge: Kaggle data has ticker symbols as column headers (wide format)
# Solution: Unpivot each metric (Open, High, Low, Close, Volume) separately,
#           then merge on (Date, Ticker) to create normalized long-format table
# Result: One row per (Date, Ticker) with OHLCV columns

import pandas as pd

# Starting point: df_pd from Kaggle
df = df_pd.copy()

# 1) Extract ticker row (row 0 contains ticker symbols)
header_row = df.iloc[0]

# 2) Nur die Datenzeilen behalten (Zeile 0 = "Ticker", Zeile 1 = "Date")
df_data = df[~df["Price"].isin(["Ticker", "Date"])].copy()

# 3) Hilfsfunktion: relevante Spalten je Kennzahl finden
def get_cols(prefix: str):
    return [c for c in df.columns if c == prefix or c.startswith(prefix + ".")]

open_cols   = get_cols("Open")
high_cols   = get_cols("High")
low_cols    = get_cols("Low")
close_cols  = get_cols("Close")
volume_cols = get_cols("Volume")

# 4) Hilfsfunktion: eine Kennzahl in Long-Format bringen
def melt_metric(metric_name: str, metric_cols):
    # neutraler value_name, um Kollision mit bestehenden Spalten zu vermeiden
    tmp = df_data.melt(
        id_vars=["Price"],
        value_vars=metric_cols,
        var_name="Column",
        value_name=f"{metric_name}_val"
    )
    # Ticker anhand der ersten Zeile zuordnen
    tmp["Ticker"] = tmp["Column"].map(lambda col: header_row[col])
    # leere Werte und Spalten ohne Ticker verwerfen
    tmp = tmp.dropna(subset=[f"{metric_name}_val", "Ticker"])
    # Kennzahl-Spalte auf den echten Namen zurückbenennen
    tmp = tmp.rename(columns={f"{metric_name}_val": metric_name})
    return tmp[["Price", "Ticker", metric_name]]

# 5) OHLCV unpivoten
open_m   = melt_metric("Open",   open_cols)
high_m   = melt_metric("High",   high_cols)
low_m    = melt_metric("Low",    low_cols)
close_m  = melt_metric("Close",  close_cols)
volume_m = melt_metric("Volume", volume_cols)

# 6) Alles auf (Price, Ticker) mergen
df_tidy = close_m.merge(open_m,   on=["Price", "Ticker"], how="left")
df_tidy = df_tidy.merge(high_m,   on=["Price", "Ticker"], how="left")
df_tidy = df_tidy.merge(low_m,    on=["Price", "Ticker"], how="left")
df_tidy = df_tidy.merge(volume_m, on=["Price", "Ticker"], how="left")

# 7) Datentypen und Datumsspalte setzen
df_tidy["Date"] = pd.to_datetime(df_tidy["Price"], errors="coerce").dt.date

for col in ["Open", "High", "Low", "Close", "Volume"]:
    df_tidy[col] = pd.to_numeric(df_tidy[col], errors="coerce")

df_tidy = (
    df_tidy
    .drop(columns=["Price"])
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

df_tidy.head()


In [0]:
# ==============================================================================
# STEP 5: Convert to Spark DataFrame for distributed processing
# ==============================================================================
# Pandas → Spark: Transition from single-node to distributed processing
# Schema inference: Automatic type detection from pandas DataFrame

df_spark = spark.createDataFrame(df_tidy)
df_spark.printSchema()
df_spark.show(5)


In [0]:
# ==============================================================================
# STEP 6: Persist to Delta Lake
# ==============================================================================
# Pattern: Full refresh (overwrite mode)
# Target: Unity Catalog table thekitchen.kg.sp500_aa_daily
# Alternative: Change to "append" for incremental loads

(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")  # Change to "append" for incremental loads
    .saveAsTable("thekitchen.kg.sp500_aa_daily")
)


In [0]:
# ==============================================================================
# STEP 7: Verify loaded data
# ==============================================================================
# Quick check: Display first 10 rows to confirm successful load

%skip
df_check = spark.table("thekitchen.kg.sp500_aa_daily")
df_check.show(10)


---

## Technical Highlights

### Design Patterns
* **Wide-to-Long Transformation**: Unpivots multi-ticker wide format to normalized long format
* **Metric-by-Metric Melt**: Processes OHLCV separately then merges for clean schema
* **Secure Credential Management**: API token via Job Parameters (not hardcoded)
* **Kaggle SDK Integration**: Direct dataset access without manual CSV downloads

### Data Transformation Complexity
* **Input**: ~500 tickers × 5 metrics (OHLCV) = ~2,500 columns per date
* **Output**: One row per (Date, Ticker) with 7 columns (Date, Ticker, OHLCV)
* **Challenge**: Header row contains ticker symbols, requires dynamic mapping
* **Solution**: Extract header mapping, melt each metric, merge on (Date, Ticker)

### Production Readiness Checklist
- [ ] Move API token to Databricks Secrets (`dbutils.secrets.get()`)
- [ ] Add data quality checks (null values, duplicate dates, outlier detection)
- [ ] Implement incremental load pattern (detect new dates only)
- [ ] Add schema validation (ensure all expected tickers present)
- [ ] Schedule via Databricks Jobs (daily after market close)
- [ ] Add alerting for missing data or API failures
- [ ] Consider partitioning by date for query performance

### Use Cases
* **Time-Series Analysis**: Historical stock price analysis across all S&P 500 stocks
* **Cross-Sectional Comparisons**: Compare metrics across tickers for specific dates
* **Technical Indicators**: Foundation for RSI, moving averages, Bollinger bands
* **Portfolio Analytics**: Track multi-asset portfolio performance

### Key Metrics
* **Coverage**: All S&P 500 constituents (historical + current)
* **Granularity**: Daily OHLCV data
* **Format**: Normalized long-format (one row per date-ticker)